# core

> Examples using `compact` dialect

### Primitive Procedures

Primitive procedures are built in and cannot be redefined — attempting `(define + ...)` raises a `SyntaxError`.

**Arithmetic**

| Operator | Meaning |
|---|---|
| `+` `-` `*` `/` | Variadic; `-` negates if unary, `/` inverts if unary |
| `=` `<` `>` `<=` `>=` | Numeric comparison |
| `abs` `min` `max` `expt` `sqrt` | Common numeric operations |
| `floor` `ceiling` `round` `truncate` | Rounding |
| `modulo` `remainder` | Integer division remainder (differ on negatives) |

**Lists**

| Operator | Meaning |
|---|---|
| `list` `cons` `car` `cdr` | Construction and access |
| `null?` `pair?` `list?` | Predicates |

**Strings**

| Operator | Meaning |
|---|---|
| `string-append` `string-length` `substring` | Operations |
| `number->string` `string->number` | Numeric conversion |
| `symbol->string` `string->symbol` | Symbol conversion |

**Predicates & equality**

| Operator | Meaning |
|---|---|
| `number?` `string?` `symbol?` `boolean?` | Type predicates |
| `equal?` | Deep structural equality (type-strict) |
| `not` | Boolean negation |

**Other**

| Operator | Meaning |
|---|---|
| `error` | Raise an exception with a message and optional irritants |

### Special Forms

Special forms look like procedure calls but do not evaluate all arguments eagerly —
each form decides when and how to evaluate its sub-expressions.

**Core**

| Form | Syntax | Meaning |
|---|---|---|
| `quote` | `'x` | Return expression unevaluated |
| `define` | `(define name val)` / `(define (f a…) body)` | Bind a symbol in the current environment |
| `lambda` | `(lambda (a…) body)` | Anonymous procedure |
| `begin` | `(begin e …)` | Evaluate sequence, return last |
| `set!` | `(set! name val)` | Mutate an existing binding |

**Control flow**

| Form | Syntax | Meaning |
|---|---|---|
| `if` | `(if test then else)` | Conditional; only the taken branch is evaluated |
| `cond` | `(cond (test expr) … (else expr))` | Multi-branch conditional |
| `when` | `(when test body…)` | Evaluate body only if test is truthy |
| `unless` | `(unless test body…)` | Evaluate body only if test is false |
| `case` | `(case val ((v…) expr)… (else expr))` | Dispatch on a value against literal lists |
| `and` | `(and e …)` | Short-circuit; returns last value or `#f` |
| `or` | `(or e …)` | Short-circuit; returns first truthy value or `#f` |

**Binding**

| Form | Syntax | Meaning |
|---|---|---|
| `let` | `(let ((x v) …) body)` | Local bindings (all evaluated in outer env) |
| `let` (named) | `(let loop ((i 0)) body)` | Named let — local recursive loop |
| `let*` | `(let* ((x v) …) body)` | Sequential bindings; each sees the previous |
| `letrec` | `(letrec ((x v) …) body)` | Recursive bindings; each can reference the others |

**Lists**

| Form | Syntax | Meaning |
|---|---|---|
| `apply` | `(apply fn arg… lst)` | Call `fn` spreading `lst` as its arguments |
| `map` | `(map fn lst…)` | Apply `fn` across one or more lists |
| `filter` | `(filter fn lst)` | Keep elements where `fn` returns truthy |
| `for-each` | `(for-each fn lst…)` | Like `map` but for side effects |
| `fold-left` | `(fold-left fn init lst)` | Reduce left-to-right with accumulator |
| `fold-right` | `(fold-right fn init lst)` | Reduce right-to-left with accumulator |

**Macros**

| Form | Syntax | Meaning |
|---|---|---|
| `macro` | `(macro (a…) body)` | Define a code transformer |
| `quasiquote` | `` `(… ,x ,@xs) `` | Template with `,` splice and `,@` list splice |

In [ ]:
from compact import lisp

In [ ]:
lisp.register_magic()

## Examples

### Core

**quote** — return unevaluated

In [ ]:
%%lisp
'(1 2 3)

[1, 2, 3]

**define** — bind a symbol; **lambda** — anonymous procedure

In [ ]:
%%lisp
(begin
  (define x 42)
  x)

42

In [ ]:
%%lisp
(begin
  (define (square x) (* x x))
  (square 5))

25

**begin** — evaluate a sequence, return the last

In [ ]:
%%lisp
(begin 1 2 3)

3

### Control flow

**if** — conditional; only the taken branch is evaluated

In [ ]:
%%lisp
(if #t 42 0)

42

**cond** — multi-branch conditional

In [ ]:
%%lisp
(cond
  ((= 1 2) "no")
  ((= 1 1) "yes")
  (else    "other"))

'yes'

**when / unless** — one-branch conditional

In [ ]:
%%lisp
(when (= 1 1) 42)

42

In [ ]:
%%lisp
(unless (= 1 2) 42)

42

**case** — dispatch on a value against literal lists

In [ ]:
%%lisp
(case (* 2 3)
  ((2 3 5 7) 'prime)
  ((1 4 6 8) 'composite)
  (else      'other))

compact.types.Symbol(s='composite')

**and / or** — short-circuit; return last evaluated value

In [ ]:
%%lisp
(and 1 2 3)

3

In [ ]:
%%lisp
(or #f #f 42)

42

In [ ]:
%%lisp
(and #f (/ 1 0))  ; short-circuits — (/ 1 0) never evaluated

False

In [ ]:
%%lisp
(or 42 (/ 1 0))   ; short-circuits — (/ 1 0) never evaluated

42

### Binding

**set!** — mutate an existing binding

In [ ]:
%%lisp
(begin
  (define x 1)
  (set! x 42)
  x)

42

**let** — local bindings

In [ ]:
%%lisp
(let ((x 40))
  (+ x 2))

42

**let*** — sequential bindings; each binding sees the previous

In [ ]:
%%lisp
(let* ((x 2) (y (* x 3)))
  (+ x y))

8

**letrec** — bindings evaluated in the new env; enables mutual recursion

In [ ]:
%%lisp
(letrec ((even? (lambda (n) (if (= n 0) #t (odd?  (- n 1)))))
         (odd?  (lambda (n) (if (= n 0) #f (even? (- n 1))))))
  (even? 10))

True

**tail calls** — deep recursion without stack overflow

**named let** — recursive loop without a top-level define

In [ ]:
%%lisp
(let loop ((i 0) (acc 0))
  (if (= i 5) acc (loop (+ i 1) (+ acc i))))

10

In [ ]:
%%lisp
(begin
  (define (fact n acc)
    (if (= n 0) acc (fact (- n 1) (* n acc))))
  (fact 1000 1))

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

### Lists

**apply** — call a function with a list as its arguments

In [ ]:
%%lisp
(apply + '(1 2 3 4))

10

**map / filter / for-each** — list processing

In [ ]:
%%lisp
(map (lambda (x) (* x x)) '(1 2 3 4 5))

[1, 4, 9, 16, 25]

In [ ]:
%%lisp
(filter (lambda (x) (> x 2)) '(1 2 3 4 5))

[3, 4, 5]

**fold-left / fold-right** — reduce a list with an accumulator

In [ ]:
%%lisp
(fold-left (lambda (acc x) (+ acc x)) 0 '(1 2 3 4 5))

15

In [ ]:
%%lisp
(fold-right (lambda (x acc) (cons x acc)) '() '(1 2 3))

[1, 2, 3]

**lists** — car, cdr, cons

In [ ]:
%%lisp
(car '(10 20 30))

10

In [ ]:
%%lisp
(cdr '(10 20 30))

[20, 30]

In [ ]:
%%lisp
(cons 1 '(2 3))

[1, 2, 3]

**error** — raise with a message and optional irritants

In [ ]:
%%lisp
(define (safe-div x y)
  (if (= y 0) (error "division by zero" x y) (/ x y)))
(safe-div 10 2)

5.0

In [ ]:
try: '(safe-div 10 0)' @ lisp
except Exception as e: assert 'division by zero' in str(e)

### Data

**string primitives** — append, length, slice, and numeric conversion

In [ ]:
%%lisp
(string-append "hello" ", " "world")

'hello, world'

In [ ]:
%%lisp
(string-length "hello")

5

In [ ]:
%%lisp
(substring "hello" 1 3)

'el'

In [ ]:
%%lisp
(number->string 42)

'42'

In [ ]:
%%lisp
(string->number "3.14")

3.14

In [ ]:
%%lisp
(string->number "not-a-number")

False

**equal?** — deep structural equality, type-strict

In [ ]:
%%lisp
(equal? '(1 2 3) '(1 2 3))

True

In [ ]:
%%lisp
(equal? '(1 2) '(1 2 3))

False

In [ ]:
%%lisp
(equal? 1 1.0)

False

### Macros & quasiquote

**macros** — transform code before evaluation

**macro** — add new syntax that is indistinguishable from built-in forms; arguments are passed unevaluated

In [ ]:
%%lisp
(begin
  (define while
    (macro (cond . body)
      `(let loop () (when ,cond ,@body (loop)))))

  (let ((i 0) (acc 0))
    (while (< i 5)
      (set! acc (+ acc i))
      (set! i (+ i 1)))
    acc))

10

**symbol->string / string->symbol** — symbol/string conversion

In [ ]:
%%lisp
(symbol->string 'hello)

'hello'

**numeric utilities** — rounding, min/max, exponentiation, modulo

In [ ]:
%%lisp
(floor 3.7)

3

In [ ]:
%%lisp
(expt 2 10)

1024

In [ ]:
%%lisp
(modulo -13 4)

3

In [ ]:
%%lisp
(remainder -13 4)

-1

In [ ]:
%%lisp
(ceiling 3.2)

4

In [ ]:
%%lisp
(round 3.5)

4

In [ ]:
%%lisp
(truncate 3.7)

3

In [ ]:
%%lisp
(abs -5)

5

In [ ]:
%%lisp
(min 3 1 2)

1

In [ ]:
%%lisp
(max 3 1 2)

3

In [ ]:
%%lisp
(string->symbol "hello")

compact.types.Symbol(s='hello')

In [ ]:
%%lisp
(string->symbol (string-append "make-" (symbol->string 'widget)))

compact.types.Symbol(s='make-widget')

**quasiquote** — template with selective splicing

In [ ]:
%%lisp
`(a ,(+ 1 2) ,@'(4 5))

[compact.types.Symbol(s='a'), 3, 4, 5]

### Python interop

**python interop** — lisp expressions can reference python symbols

In [ ]:
a = 5
"""
(+ a 6)
""" @ lisp

11

In [ ]:
"""
(define (factorial n acc)
  (if (= n 0) acc
      (factorial (- n 1) (* n acc))))
""" @ lisp

# call this factorial procedure from python
r = factorial(10000, 1)  # works fine; Python would hit recursion limit

# too many characters to print so print number of digits instead
import math

math.floor(r.bit_length() * math.log10(2)) + 1


35660

In [ ]:
%%time
import math
m10k = math.factorial(10000)

CPU times: user 1.9 ms, sys: 0 ns, total: 1.9 ms


Wall time: 1.91 ms


In [ ]:
%%time
s10k = factorial(10000, 1)

CPU times: user 294 ms, sys: 0 ns, total: 294 ms


Wall time: 292 ms


In [ ]:
s10k == m10k

True

**accessing lisp symbols from python** — use `lisp[name]` to look up any lisp symbol, including ones with names that are not valid python identifiers

In [ ]:
lisp["null?"]([])           # True
lisp["string->number"]("42") # 42
lisp["+"](1, 2, 3)           # 6

6

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()